# 🎓 IA Conseiller d'Orientation — Leçon 1 : Les fondations

Bienvenue ! Dans cette première leçon, nous commençons à construire une **IA conseillère d'orientation** : un programme qui comprend ce qu'un étudiant écrit sur ses goûts et lui propose des métiers et des formations.

**Ce que vous allez apprendre aujourd'hui :**
1. Comment une IA comprend le texte
2. Votre premier modèle Hugging Face
3. Créer un petit jeu de données d'orientation
4. Faire des prédictions avec un modèle pré-entraîné

### 🤖 Le lien avec notre parcours IA et robotique
Ce notebook paraît loin des robots, mais c'est **la même chaîne de travail** :

| Dans un robot | Dans ce notebook |
|---|---|
| Un capteur (caméra, ultrason) produit des **nombres** | Le texte est converti en **nombres** (tokens) |
| Un réseau de neurones **perçoit** l'environnement | Un modèle **perçoit** le sens d'une phrase |
| On collecte des données, on entraîne, on teste | Données → modèle → entraînement → test |
| Un modèle pré-entraîné est adapté à notre robot | Un modèle pré-entraîné sera adapté à notre contexte (fine-tuning) |

Plus tard, un robot qui comprend des commandes ou des retours d'utilisateurs en français utilisera exactement ces outils.

Commençons depuis le **tout début** !

## 📦 Partie 1 : Préparer votre environnement

Dans Google Colab, `transformers` est déjà installé. La cellule suivante installe quand même les paquets nécessaires (30 à 60 secondes la première fois).

In [ ]:
# Installation des paquets (Google Colab)
!pip install -q transformers datasets accelerate sentencepiece

In [ ]:
import pandas as pd
from transformers import pipeline, AutoTokenizer
print("✅ Tous les paquets sont chargés avec succès !")

## 📖 Partie 2 : Comment l'IA comprend-elle le texte ?

Les humains lisent des mots. **L'IA lit des NOMBRES.**

Exemple :
- L'humain voit : `"J'adore programmer des robots"`
- L'IA voit : une liste de nombres, appelés **TOKENS** (identifiants)

Le processus :
1. **TEXTE → TOKENS** (découpage en morceaux)
2. **TOKENS → NOMBRES** (conversion en identifiants)
3. **NOMBRES → EMBEDDINGS** (représentation mathématique du sens)
4. **EMBEDDINGS → COMPRÉHENSION** (le modèle calcule une réponse)

> 🔧 **Analogie robotique :** un capteur de distance ne « comprend » pas un mur, il fournit un nombre (par exemple 0,35 m). Le texte subit le même traitement : tout devient des nombres avant d'entrer dans un réseau de neurones.

Voyons cela en vrai. Le modèle de cette leçon sera **multilingue** (il connaît le français). Nous chargeons d'abord son *tokenizer* :

In [ ]:
MODELES_CANDIDATS = [
    "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    # Copie identique avec un tokenizer rapide préconstruit :
    # sert de secours si la version de transformers a un problème avec l'original.
    "NetworkIsLife/twitter-xlm-roberta-base-sentiment-fast",
]

MODELE = None
tokenizer = None
for nom in MODELES_CANDIDATS:
    try:
        tokenizer = AutoTokenizer.from_pretrained(nom)
        MODELE = nom
        print(f"✅ Tokenizer chargé depuis : {nom}")
        break
    except Exception as e:
        print(f"⚠️ Échec avec {nom} : {e}")

if tokenizer is None:
    print("❌ Impossible de charger le tokenizer. Vérifiez votre connexion internet.")

In [ ]:
phrases = [
    "J'adore programmer des robots",
    "Mon capteur ultrason donne des valeurs fausses",
]

for phrase in phrases:
    tokens = tokenizer.tokenize(phrase)
    identifiants = tokenizer.convert_tokens_to_ids(tokens)
    print(f"Phrase       : {phrase}")
    print(f"Tokens       : {tokens}")
    print(f"Identifiants : {identifiants}\n")

print("💡 Le symbole ▁ marque le début d'un mot. Un mot rare peut être découpé en plusieurs morceaux.")

## 🤖 Partie 3 : Votre premier modèle d'IA

Nous chargeons un modèle d'**analyse de sentiments**. Il détecte si un texte est **positif**, **neutre** ou **négatif**.

**Pourquoi commencer par là ?** Parce que notre conseiller d'orientation doit :
- comprendre si un étudiant est **INTÉRESSÉ** par quelque chose (positif),
- comprendre s'il **n'aime pas** quelque chose (négatif).

> ⚠️ **Pourquoi ce modèle et pas celui de la version anglaise ?** Le modèle `distilbert-base-uncased-finetuned-sst-2-english` ne comprend que l'anglais. Comme nos phrases sont maintenant en français, nous utilisons un modèle **multilingue** (XLM-RoBERTa), ajusté entre autres sur le français. Il est plus lourd (environ 1 Go à télécharger) : le premier chargement peut prendre une à deux minutes.

In [ ]:
print("⏳ Chargement du modèle (cela peut prendre un peu de temps)...")
analyseur = pipeline("sentiment-analysis", model=MODELE, tokenizer=tokenizer)
print("✅ Modèle chargé avec succès !")

### 🧪 Tester le modèle

Nous testons des phrases d'étudiants **et** des phrases liées à la robotique.

In [ ]:
phrases_test = [
    "J'adore travailler avec les ordinateurs et programmer",
    "Je déteste les mathématiques, les chiffres m'ennuient",
    "La biologie et aider les gens me rendent heureux",
    "Je n'ai pas envie de rester assis devant un bureau toute la journée",
    "J'adore programmer des robots",
    "Le robot a évité l'obstacle sans problème, c'est génial !",
    "Mon capteur ultrason donne des valeurs fausses, je suis très frustré",
]

# Selon la version du modèle, les étiquettes peuvent s'appeler "Positive" ou "LABEL_2"...
NORMALISATION = {
    "positive": "POSITIF", "label_2": "POSITIF",
    "neutral": "NEUTRE",   "label_1": "NEUTRE",
    "negative": "NÉGATIF", "label_0": "NÉGATIF",
}
EMOJIS = {"POSITIF": "😊", "NEUTRE": "😐", "NÉGATIF": "😟"}

print("Voyons comment l'IA interprète ces phrases :\n")
for phrase in phrases_test:
    resultat = analyseur(phrase)[0]
    etiquette = NORMALISATION.get(resultat["label"].lower(), resultat["label"])
    print(f"{EMOJIS.get(etiquette, '❓')} « {phrase} »")
    print(f"   → {etiquette} (confiance : {resultat['score']:.2%})\n")

### 💡 Observation et limites

Lisez les résultats et posez-vous ces questions :
- Le modèle a-t-il bien interprété **chaque** phrase ?
- Que dit-il de « Je n'ai pas envie de rester assis devant un bureau toute la journée » ? Cette phrase indique-t-elle un goût pour un *métier* précis ?

**Ce qu'il faut retenir :** l'analyse de sentiments dit **si** la personne est positive ou négative, mais **pas de quoi** elle parle. Elle ne suffit donc pas pour recommander un métier. C'est pour cela que nous allons **entraîner (fine-tuner)** un modèle sur nos propres données d'orientation dans les prochaines leçons.

> Un modèle n'est jamais parfait. En robotique aussi, un modèle de perception se trompe parfois : on mesure toujours ses erreurs avant de lui faire confiance.

## 📊 Partie 4 : Créer votre jeu de données d'orientation

Pour le fine-tuning, il nous faut des **DONNÉES** : des exemples associant

**intérêts / compétences de l'étudiant → métiers recommandés → formations suggérées**

Créons ensemble un petit jeu de données. Dans un vrai projet, vous auriez des centaines ou des milliers d'exemples.

> 🇭🇹 **Adapté à notre contexte :** au lieu de grandes universités étrangères, nous proposons des **formations accessibles à des étudiants haïtiens** (ESIH, UEH, Université Quisqueya, formations en ligne) et nous avons ajouté des parcours **robotique, systèmes embarqués et IA**.
>
> ⚠️ **À vérifier et à adapter par le formateur :** les noms d'établissements et de facultés sont des exemples. Vérifiez leur exactitude et ajoutez les formations que vous connaissez dans votre région.

In [ ]:
jeu_orientation = {
    "entree_etudiant": [
        "J'adore programmer et créer des sites web",
        "J'aime résoudre des problèmes de maths et travailler avec des données",
        "J'aime aider les gens et comprendre la psychologie",
        "Je suis passionné par l'art et le design créatif",
        "J'adore la biologie et je veux aider à guérir les maladies",
        "J'aime écrire des histoires et créer du contenu",
        "J'aime construire des choses et comprendre comment fonctionnent les machines",
        "Je m'intéresse aux affaires et à l'entrepreneuriat",
        "J'adore enseigner et expliquer des notions aux autres",
        "J'aime la musique et je veux créer des bandes sonores",
        # --- Parcours robotique, embarqué et IA ---
        "Je suis fasciné par les robots et par leur capacité à se déplacer seuls",
        "J'aime souder des circuits et programmer des microcontrôleurs",
        "Je veux que les machines apprennent et prennent des décisions",
        "Je veux utiliser la technologie pour améliorer la santé en Haïti",
    ],
    "metiers_recommandes": [
        "Développeur logiciel, Développeur web, Ingénieur full-stack",
        "Data scientist, Statisticien, Analyste financier",
        "Psychologue, Conseiller, Travailleur social",
        "Graphiste, Designer UI/UX, Illustrateur",
        "Chercheur biomédical, Médecin, Pharmacien",
        "Rédacteur, Journaliste, Auteur",
        "Ingénieur mécanicien, Ingénieur civil, Architecte",
        "Analyste d'affaires, Entrepreneur, Responsable marketing",
        "Enseignant, Professeur, Créateur de contenus pédagogiques",
        "Producteur de musique, Ingénieur du son, Compositeur",
        "Ingénieur en robotique, Ingénieur en systèmes embarqués, Technicien en automatisation",
        "Ingénieur en électronique, Développeur de systèmes embarqués, Technicien IoT",
        "Ingénieur en intelligence artificielle, Ingénieur en apprentissage automatique, Chercheur en IA",
        "Ingénieur biomédical, Développeur de logiciels de santé, Informaticien de santé",
    ],
    "formations_recommandees": [
        "ESIH (informatique), Université Quisqueya, formations en ligne (freeCodeCamp, OpenClassrooms)",
        "UEH (Faculté des Sciences), ESIH, cours en ligne (Coursera, edX)",
        "UEH (Sciences Humaines), Université Quisqueya",
        "Formations en ligne en design (Domestika, Coursera) et création d'un portfolio",
        "UEH (Faculté de Médecine et de Pharmacie), formations en sciences de la santé",
        "Formations en communication et journalisme, ateliers d'écriture",
        "UEH (Faculté des Sciences), écoles d'ingénieurs",
        "UEH (INAGHEI), Université Quisqueya",
        "UEH (École Normale Supérieure)",
        "Écoles de musique, formations en ligne en production musicale",
        "ESIH (infotronique), UEH (Faculté des Sciences), ROS 2 et robotique en ligne",
        "ESIH (infotronique), formations en électronique et microcontrôleurs",
        "ESIH, cours en ligne d'IA (Hugging Face, Coursera, fast.ai)",
        "ESIH, UEH (Faculté de Médecine et de Pharmacie), projets de santé numérique",
    ],
}

# Conversion en DataFrame (pensez à un tableau Excel)
df = pd.DataFrame(jeu_orientation)

print("📋 Voici notre jeu de données :\n")
print(df.to_string(index=False))

print(f"\n✅ Nous avons {len(df)} exemples d'entraînement !")
print("Ceci est un TOUT PETIT jeu de données pour apprendre. Un vrai modèle demande 100 à 10 000 exemples ou plus.")

# Sauvegarde du jeu de données
df.to_csv("jeu_orientation.csv", index=False)
print("\n💾 Jeu de données sauvegardé sous « jeu_orientation.csv »")

## 🎯 Mini-défi n°1

**VOTRE MISSION :** ajouter **5 exemples de plus** au jeu de données, en suivant le même modèle :
1. Un intérêt ou une compétence de l'étudiant
2. Des métiers recommandés
3. Des formations suggérées

Pensez à différents domaines : sport, environnement, agriculture, technologie, arts, santé, énergie…

**INDICE :** pensez à ce qui VOUS intéressait, ou à ce qu'aiment vos amis. Qu'est-ce qui existe vraiment autour de vous en Haïti ?

**Pour réussir ce défi :**
1. Complétez les listes dans la cellule ci-dessous (gardez le même nombre d'éléments dans les trois listes)
2. Exécutez la cellule pour voir votre jeu de données agrandi

Pas de triche : essayez avant de demander la réponse ! 😉

In [ ]:
# 👉 À vous de jouer : ajoutez 5 éléments dans CHAQUE liste (même ordre, même nombre).
mes_nouveaux_exemples = pd.DataFrame({
    "entree_etudiant": [
        # "J'aime ...",
    ],
    "metiers_recommandes": [
        # "Métier 1, Métier 2, Métier 3",
    ],
    "formations_recommandees": [
        # "Formation 1, Formation 2",
    ],
})

df_etendu = pd.concat([df, mes_nouveaux_exemples], ignore_index=True)
print(f"Taille du jeu de données : {len(df_etendu)} exemples (objectif : {len(df) + 5})")
df_etendu.tail(7)

## 🎓 Leçon terminée !

**CE QUE VOUS AVEZ APPRIS AUJOURD'HUI :**
- ✅ Comment l'IA convertit le texte en nombres (tokenisation)
- ✅ Comment charger un modèle pré-entraîné depuis Hugging Face
- ✅ Comment tester un modèle avec de vrais exemples, et voir ses limites
- ✅ Comment créer un jeu de données d'entraînement
- ✅ La structure nécessaire pour un conseiller d'orientation

**AU PROGRAMME DE LA LEÇON 2 :**
- 📚 Comprendre le processus de fine-tuning
- 🔧 Préparer les données au bon format
- 🚀 Entraîner votre premier modèle
- 💾 Sauvegarder et recharger votre modèle personnalisé

**DEVOIRS (FACULTATIFS MAIS RECOMMANDÉS) :**
1. Terminez le mini-défi ci-dessus.
2. Réfléchissez aux autres informations utiles pour recommander un métier (notes ? activités extrascolaires ? traits de personnalité ?).
3. Explorez Hugging Face : https://huggingface.co/models
   - Cherchez différents modèles
   - Lisez ce qu'ils font
   - Familiarisez-vous avec le site
4. **Défi bonus (curieux de robotique) :** écrivez 5 phrases de retour d'utilisateur sur un robot (« le robot est lent », « la caméra reconnaît bien les objets »…) et regardez ce que dit le modèle. Que se passerait-il avec des phrases en créole haïtien ? (Le modèle n'a pas été ajusté sur cette langue : comparez et notez vos observations.)

Des questions ? Notez-les, nous y répondrons dans la Leçon 2 !

## 📝 Notes du formateur (pour votre compréhension)

**CONCEPTS CLÉS À RETENIR :**

1. **MODÈLES PRÉ-ENTRAÎNÉS**
   - Ils ont déjà appris à partir de millions d'exemples.
   - Nous les adaptons à NOTRE tâche (orientation).
   - C'est comme embaucher un enseignant expérimenté et le former à VOTRE programme.
   - En robotique : même idée quand on réutilise un modèle de vision déjà entraîné sur des millions d'images.

2. **FINE-TUNING**
   - Prendre un modèle pré-entraîné.
   - Continuer son entraînement sur VOS données.
   - Le spécialiser pour VOTRE tâche.

3. **LA CHAÎNE DE TRAVAIL**
   Collecte des données → Préparation des données → Choix du modèle → Fine-tuning → Test

   Nous en sommes à : Collecte des données ✅
   Prochaine étape : Préparation des données

4. **POINT D'ATTENTION SUR LES DONNÉES**
   Un modèle reflète ses données. Avec 14 exemples, il sera très limité et biaisé vers ce que nous avons écrit. Insistez sur ce point : plus de données, et de meilleures données, donnent un meilleur modèle (comme pour un robot entraîné sur trop peu de situations).

5. **CHANGEMENT PAR RAPPORT À LA VERSION ANGLAISE**
   Le modèle de sentiment anglais (DistilBERT) a été remplacé par un modèle multilingue, car il ne comprend pas le français. Testez le notebook entièrement dans Colab avant le cours : le téléchargement du modèle demande de la connexion internet.

À bientôt pour une prochaine Leçon!